# 02 · Autoencoder, thresholds and evaluation
WBS 2, 3 and 4. Uses the saved model and results from `python run_all.py`.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..')); os.chdir('..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import config, analysis
from detector import Detector, holdout
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix

In [ ]:
det = Detector()
clean = pd.read_csv(config.CLEAN_DATA_FILE, parse_dates=['timestamp'])
res = analysis.add_context(det.detect(clean))
h = holdout(res)
print('alert threshold', round(det.threshold, 3))

## 2.3.3 / 3.1 Reconstruction errors and ranked events

In [ ]:
top, by_type = analysis.suspicious(res, 15)
top.round(2)

## 3.2 Error distribution and threshold

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for y, c in [(0, 'grey'), (1, 'red')]:
    ax.hist(np.log10(h[h.is_anomaly == y].score.clip(lower=1e-3)), bins=70, alpha=.6, color=c, label=['normal', 'error'][y])
ax.axvline(np.log10(det.threshold), color='teal', lw=2, label='alert threshold'); ax.set_yscale('log'); ax.legend(); ax.set_xlabel('log10 anomaly score'); plt.show()

In [ ]:
pd.DataFrame(json.load(open(config.THRESHOLDS_FILE))['evaluation'])[['name', 'threshold', 'alerts', 'precision', 'recall', 'f1']].round(3)

## 4.1.1 Detection metrics: precision, recall, F1, confusion matrix, ROC-AUC, PR-AUC

In [ ]:
print(analysis.comparison_table(h).round(3).to_string(index=False))
m = json.load(open(config.METRICS_FILE)); print('ROC-AUC', round(m['roc_auc'], 3), ' PR-AUC', round(m['pr_auc'], 3))
confusion_matrix(h.is_anomaly, h.flagged)

In [ ]:
fpr, tpr, _ = roc_curve(h.is_anomaly, h.score); p, r, _ = precision_recall_curve(h.is_anomaly, h.score)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5)); ax[0].plot(fpr, tpr); ax[0].set_title('ROC'); ax[1].plot(r, p); ax[1].set_title('Precision-recall'); plt.show()

## 4.1.2 False positives and their causes

In [ ]:
fa = analysis.fp_analysis(h, det.threshold)
print(*fa['summary'], sep='\n'); analysis.fp_causes(h)

## 4.1.3 Sensitivity: how the threshold changes what is flagged

In [ ]:
rows = []
for s in np.arange(.25, 3.01, .25):
    f = (h.score > det.threshold / s)
    tp = (f & (h.is_anomaly == 1)).sum(); rows.append((s, f.sum(), tp / max(f.sum(), 1), tp / h.is_anomaly.sum()))
pd.DataFrame(rows, columns=['sensitivity', 'alerts', 'precision', 'recall']).round(3)

## 4.2 Normal vs anomalous events, cross-channel inconsistencies, limitations

In [ ]:
analysis.cross_channel(res).round(3)

In [ ]:
pd.read_csv(config.HISTORY_FILE).round(3)  # how much history the model needs